# Kalliope SRU Abfrage und parsen der Briefwechsel von Werner Heisenberg
Sources: Code f+r die SRU-Abfrage und das Parsen der Daten adaptiert nach DNB SRU Tutorial:

https://github.com/deutsche-nationalbibliothek/dnblab/blob/main/DNB_SRU_Tutorial.ipynb 

Dokumentation Kalliope SRU:
https://kalliope-verbund.info/de/support/sru.html

Zweck ist es, die Daten so auszulesen, dass im Anschluß für eine Netzwerkanalyse weiterverarbeitet werden können

In [18]:
# Import necessary libraries
# Benötigte Bibliotheken importieren

import requests
from lxml import etree
import pandas as pd

In [19]:
# Function to send a SRU request to the Kalliope system with a custom query
# Funktion zum Senden einer SRU-Anfrage an das Kalliope-System mit einer benutzerdefinierten Abfrage
# SRU query
def kalliope_sru(query):
    base_url = "https://kalliope-verbund.info/sru"
    params = {
        'version': '1.2',
        'operation': 'searchRetrieve',
        'recordSchema': 'mods37',
        'maximumRecords': '100',   #mehr records werden gefunden, wenn maximum records hochgesetzt wird.
        'query': query
    }
    
    r = requests.get(base_url, params=params)
    mods_content = r.content
    records_mods = etree.fromstring(mods_content)
    
    # Check if more than 100 records
    if len(records_mods.xpath("//srw:record", namespaces={'srw': 'http://www.loc.gov/zing/srw/'})) < 100:
        return records_mods
    else:
        num_results = 100
        i = 101
        while num_results == 100:
            params.update({'startRecord': i})
            r = requests.get(base_url, params=params)
            new_mods_content = r.content
            new_records_mods = etree.fromstring(new_mods_content)
            records_mods.extend(new_records_mods.xpath("//srw:record", namespaces={'srw': 'http://www.loc.gov/zing/srw/'}))
            i += 100
            num_results = len(new_records_mods.xpath("//srw:record", namespaces={'srw': 'http://www.loc.gov/zing/srw/'}))
        
        return records_mods


In [16]:
# Function to parse a MODS record (SRU response XML) and extract relevant fields
# Funktion zum Parsen eines MODS-Datensatzes (SRU XML-Antwort) und Extraktion relevanter Felder
def parse_mods(record):
    ns = {
        'srw': 'http://www.loc.gov/zing/srw/',  # SRW namespace
        'mods': 'http://www.loc.gov/mods/v3'    # MODS namespace
    }
    
    # Extract RecordID
    recordIdentifier = record.xpath(".//mods:mods/mods:recordInfo/mods:recordIdentifier", namespaces=ns)
    recordIdentifier = recordIdentifier[0].text if recordIdentifier else "unknown"
    
    # Extract Title
    title = record.xpath(".//mods:mods/mods:titleInfo/mods:title", namespaces=ns)
    title = title[0].text if title else "unknown"
    
    # Extract Date
    date = record.xpath(".//mods:mods/mods:originInfo/mods:dateCreated", namespaces=ns)
    date = date[0].text if date else "unknown"
    
    # Extract Names and Roles
    names = record.xpath(".//mods:mods/mods:name", namespaces=ns)
    senders = []
    receivers = []
    mentioned = []
    
    for name in names:
        name_text = name.xpath(".//mods:namePart/text()", namespaces=ns)
        role_text = name.xpath(".//mods:role/mods:roleTerm[@type='text']/text()", namespaces=ns)
        
        if name_text and role_text:
            name_value = name_text[0]
            role_value = role_text[0].lower()
            
            # Categorize based on role
            if "verfasser" in role_value:  # Adjust to match actual role values
                senders.append(name_value)
            elif "adressat" in role_value:
                receivers.append(name_value)
            elif "erwähnt" in role_value:
                mentioned.append(name_value)
    
    # Extract Genre
    genre_letter = record.xpath(".//mods:mods/mods:genre[text()='Brief']", namespaces=ns)
    genre_letter = genre_letter[0].text if genre_letter else "unknown"
    
    # Return a dictionary to build the DataFrame
    return {
        "recordIdentifier": recordIdentifier,
        "title": title,
        "date": date,
        "senders": " /".join(senders),    # Combine names into a single string
        "receivers": "/ ".join(receivers),
        "mentioned": "/ ".join(mentioned),
        "genre": genre_letter
    }


In [20]:
# Define the SRU query string, e.g. to retrieve letters from a specific archive or person
# Definieren der SRU-Abfragezeichenkette, z. B. um Briefe aus einem bestimmten Archiv oder von bestimmten Personen abzurufen
# Example query

#query = 'ead.archdesc.id="DE-611-BF-73161"'

#query = 'ead.archdesc.id="DE-611-BF-73161" and ead.unitdate_end<1980'

query = 'ead.archdesc.id="DE-611-BF-73161" AND ead.unitdate_start>=1945 AND ead.unitdate_end<=1950'

#'ead.archdesc.id'
#query = "ead.addressee"=="Heisenberg"
records_xml = kalliope_sru(query)

print(f'{len(records_xml.xpath("//srw:record", namespaces={"srw": "http://www.loc.gov/zing/srw/"}))} Ergebnisse gefunden')


200 Ergebnisse gefunden


In [21]:
# Parse the retrieved XML records and convert them into a list of dictionaries
# Parsen der abgerufenen XML-Datensätze und Umwandlung in eine Liste von Dictionaries
# Parse data and convert to DataFrame
records = records_xml.xpath("//srw:record", namespaces={"srw": "http://www.loc.gov/zing/srw/"})
output = [parse_mods(record) for record in records]
df = pd.DataFrame(output)
df


,recordIdentifier,title,date,senders,receivers,mentioned,genre
0,DE-611-HS-3756352,"Brief von Werner Heisenberg an Yoshio Nishina,...",1950-11-03,"Heisenberg, Werner (1901-1976)","Nishina, Yoshio",,Brief
1,DE-611-HS-3756487,"Brief von Fr. Normann an Werner Heisenberg, 20...",1947-07-20,"Normann, Fr. [vermutlich]","Heisenberg, Werner (1901-1976)",,Brief
2,DE-611-HS-3756490,"Brief von Werner Heisenberg an Fr. Normann, 02...",1947-07-02,"Heisenberg, Werner (1901-1976)","Normann, Fr. [vermutlich]",,Brief
3,DE-611-HS-3750246,Brief von José María Otero an Werner Heisenber...,1950-09-26,"Otero, José María","Heisenberg, Werner (1901-1976)",,Brief
4,DE-611-HS-3750248,Brief von Carlos Sanchez del Rio und José Marí...,1950-08-12,"Sanchez del Rio, Carlos (1924-2013) /Otero, Jo...","Heisenberg, Werner (1901-1976)",,Brief
...,...,...,...,...,...,...,...
195,DE-611-HS-3641955,"Brief von Bruno Touschek an Werner Heisenberg,...",1948-04-01,"Touschek, Bruno (1921-1978)","Heisenberg, Werner (1901-1976)",,Brief
196,DE-611-HS-3642093,Brief von Werner Heisenberg von Cavendish Labo...,1948-02-23,"Heisenberg, Werner (1901-1976) /Cavendish Labo...","Touschek, Bruno (1921-1978)",,Brief
197,DE-611-HS-3642115,"Brief von Bruno Touschek an Werner Heisenberg,...",1948-02-20,"Touschek, Bruno (1921-1978)","Heisenberg, Werner (1901-1976)",,Brief
198,DE-611-HS-3642133,Brief von Bruno Touschek von University of Gla...,1948-01-31,"Touschek, Bruno (1921-1978) /University of Gla...","Heisenberg, Werner (1901-1976)",,Brief


In [1]:
# Optional: print raw XML for inspection
# Optional: Rohes XML zur Überprüfung ausgeben
#print(etree.tostring(records_xml, pretty_print=True).decode())

In [22]:
# Save full parsed data as CSV file
# Gespeicherte, vollständig geparste Daten als CSV-Datei
df.to_csv("heisenberg_1945-1950.csv", index=False) # adjust filename according to query

In [23]:
#df_bibsonomy_Europa_publications = df_bibsonomy_Europa.loc[df_bibsonomy_Europa['type'] == 'Publication']
df_B = df.loc[df["genre"] == "Brief"]
df_B

,recordIdentifier,title,date,senders,receivers,mentioned,genre
0,DE-611-HS-3756352,"Brief von Werner Heisenberg an Yoshio Nishina,...",1950-11-03,"Heisenberg, Werner (1901-1976)","Nishina, Yoshio",,Brief
1,DE-611-HS-3756487,"Brief von Fr. Normann an Werner Heisenberg, 20...",1947-07-20,"Normann, Fr. [vermutlich]","Heisenberg, Werner (1901-1976)",,Brief
2,DE-611-HS-3756490,"Brief von Werner Heisenberg an Fr. Normann, 02...",1947-07-02,"Heisenberg, Werner (1901-1976)","Normann, Fr. [vermutlich]",,Brief
3,DE-611-HS-3750246,Brief von José María Otero an Werner Heisenber...,1950-09-26,"Otero, José María","Heisenberg, Werner (1901-1976)",,Brief
4,DE-611-HS-3750248,Brief von Carlos Sanchez del Rio und José Marí...,1950-08-12,"Sanchez del Rio, Carlos (1924-2013) /Otero, Jo...","Heisenberg, Werner (1901-1976)",,Brief
...,...,...,...,...,...,...,...
195,DE-611-HS-3641955,"Brief von Bruno Touschek an Werner Heisenberg,...",1948-04-01,"Touschek, Bruno (1921-1978)","Heisenberg, Werner (1901-1976)",,Brief
196,DE-611-HS-3642093,Brief von Werner Heisenberg von Cavendish Labo...,1948-02-23,"Heisenberg, Werner (1901-1976) /Cavendish Labo...","Touschek, Bruno (1921-1978)",,Brief
197,DE-611-HS-3642115,"Brief von Bruno Touschek an Werner Heisenberg,...",1948-02-20,"Touschek, Bruno (1921-1978)","Heisenberg, Werner (1901-1976)",,Brief
198,DE-611-HS-3642133,Brief von Bruno Touschek von University of Gla...,1948-01-31,"Touschek, Bruno (1921-1978) /University of Gla...","Heisenberg, Werner (1901-1976)",,Brief


In [24]:
# Optional: save only letter-related records
# Optional: nur briefbezogene Datensätze speichern
df_B.to_csv("heisenberg_1945-1950_lettersonly.csv", index=False) # adjust filename according to query

In [25]:
# ToDo: further clean and filter the data (e.g., by sender/recipient, date)
# Noch zu tun: Daten weiter bereinigen und filtern (z. B. nach Absender/Empfänger, Datum)
# next steps to clean data: pick columns date, senders, receivers from df and think
# about how to deal with separators in order to cleary separate the columns
#df_bibsonomy_Europa_selection = df_bibsonomy_Europa_publications[
#   ["type", "id", "tags", "label", "user", "description", "date", "authors", "publisher", "isbn"]] 


df_l_selec = df_B[["date","senders", "receivers"]]
df_l_selec

,date,senders,receivers
0,1950-11-03,"Heisenberg, Werner (1901-1976)","Nishina, Yoshio"
1,1947-07-20,"Normann, Fr. [vermutlich]","Heisenberg, Werner (1901-1976)"
2,1947-07-02,"Heisenberg, Werner (1901-1976)","Normann, Fr. [vermutlich]"
3,1950-09-26,"Otero, José María","Heisenberg, Werner (1901-1976)"
4,1950-08-12,"Sanchez del Rio, Carlos (1924-2013) /Otero, Jo...","Heisenberg, Werner (1901-1976)"
...,...,...,...
195,1948-04-01,"Touschek, Bruno (1921-1978)","Heisenberg, Werner (1901-1976)"
196,1948-02-23,"Heisenberg, Werner (1901-1976) /Cavendish Labo...","Touschek, Bruno (1921-1978)"
197,1948-02-20,"Touschek, Bruno (1921-1978)","Heisenberg, Werner (1901-1976)"
198,1948-01-31,"Touschek, Bruno (1921-1978) /University of Gla...","Heisenberg, Werner (1901-1976)"


In [26]:
# Save final selection with names and dates to a CSV file
# Speichere finale Auswahl mit Namen und Daten in eine CSV-Datei

df_l_selec.to_csv("heisenberg_namesdates_1945-1950.csv", index=False, sep=";") # adjust filename according to query